# 🦾 MuJoCo Playground: LEAP Hand Cube Reorient (JAX/MJX RL)

Train a deep reinforcement learning policy for in-hand cube reorientation on **LEAP Hand v1** using **MuJoCo MJX & Brax PPO** in ~20-30 minutes on GPU.

**Step 1:** Go to `Runtime` -> `Change runtime type` -> select **GPU** (T4, A100, or L4).

In [ ]:
# 1. Install MuJoCo Playground, Brax, and dependencies
!pip install -q git+https://github.com/google-deepmind/mujoco_playground.git
!pip install -q brax flax orbax-checkpoint torch

In [ ]:
# 2. Verify GPU Acceleration
import jax
print(f"JAX Devices: {jax.devices()}")
print(f"JAX Default Backend: {jax.default_backend()}")

In [ ]:
# 3. Import MuJoCo Playground & Brax PPO
import functools
import time
import numpy as np
import flax
import os
from mujoco_playground import registry, wrapper
from mujoco_playground.config import manipulation_params
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

env_name = "LeapCubeReorient"
env_cfg = registry.get_default_config(env_name)
env = registry.load(env_name, config=env_cfg)

rl_config = manipulation_params.brax_ppo_config(env_name)
ppo_training_params = dict(rl_config)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in rl_config:
    del ppo_training_params["network_factory"]
    network_factory = functools.partial(
        ppo_networks.make_ppo_networks, **rl_config.network_factory
    )
print(f"Loaded environment: {env_name} and configured network factory")

In [ ]:
# 4. Train with JAX PPO (~20-25 mins on GPU)
t0 = time.time()
def progress_fn(num_steps, metrics):
    elapsed = time.time() - t0
    reward = metrics.get('eval/episode_reward', metrics.get('training/reward', 0.0))
    print(f"Steps: {num_steps:10d} | Elapsed: {elapsed/60:.1f}m | Reward: {reward:.2f}")

print("Starting parallel GPU training...")
make_inference_fn, params, metrics = ppo.train(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
    progress_fn=progress_fn,
    network_factory=network_factory,
    **ppo_training_params,
)
print(f"\nTraining complete in {(time.time() - t0)/60:.2f} mins!")

In [ ]:
# 5. Export Policy Weights & Download
def extract_weights(flax_params):
    flat = flax.traverse_util.flatten_dict(flax_params, sep="/")
    return {k: np.array(v) for k, v in flat.items()}

numpy_weights = extract_weights(params[0] if isinstance(params, (tuple, list)) else params)

# Save as npz and PyTorch pt
npz_file = "leap_cube_reorient_jax.npz"
np.savez(npz_file, **numpy_weights)
print(f"Saved: {npz_file}")

try:
    import torch
    pt_file = "leap_cube_reorient_jax.pt"
    torch.save({"framework": "mujoco_playground", "weights": {k: torch.from_numpy(v) for k, v in numpy_weights.items()}}, pt_file)
    print(f"Saved: {pt_file}")
except Exception as e:
    pass

from google.colab import files
files.download(npz_file)
if os.path.exists("leap_cube_reorient_jax.pt"):
    files.download("leap_cube_reorient_jax.pt")